# Avaliador de Vendas — aprenda executando (Run all)

Derivado do `avaliador_de_redacao.ipynb`: rubrica 4 etapas + LLM-as-judge + LangGraph + early-exit.

**O que você vai aprender:** StateGraph, prompt `Pontuação:`, regex, média ponderada, fallback de LLM.

## Nunca usou Colab? (30 segundos)
- **Célula de texto** (como esta): só leitura.
- **Célula de código**: tem botão ▶ à esquerda. Clique ou use `Ctrl+Enter`.
- **Run all**: menu `Tempo de execução → Executar tudo`. É o modo recomendado aqui.
- **Chaves**: na 1ª célula de código vamos pedir suas chaves via digitação segura (não aparece na tela e NÃO salva no arquivo). Você precisa de pelo menos 1: `GROQ_API_KEY` (mais rápido, recomendado), `OLLAMA_CLOUD_API_KEY` ou `OPENROUTER_API_KEY`.
- Onde criar: console.groq.com, ollama.com, openrouter.ai (planos gratuitos funcionam).

In [ ]:
# 0) Colab via GitHub: clona ou ATUALIZA o repo (evita código velho em cache)
from pathlib import Path
if not Path('src').exists() and not Path('avaliador-vendas').exists():
    print('Clonando repo...')
    !git clone https://github.com/fcervan/avaliador-vendas.git
if Path('avaliador-vendas').exists() and not Path('src').exists():
    %cd avaliador-vendas
# atualiza para a versão mais nova do GitHub (mostra o commit antes/depois)
if Path('.git').exists():
    !git log --oneline -1
    !git fetch --all && git reset --hard origin/main && git log --oneline -1
    !rm -rf src/__pycache__
print('cwd:', Path.cwd())
print('src existe:', Path('src').exists())


In [ ]:
# 0) Instala dependências (funciona local e Colab)
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# se abriu o notebook dentro de notebooks/, sobe 1 nível
if (ROOT / '../requirements.txt').exists():
    ROOT = (ROOT / '..').resolve()
print('ROOT:', ROOT)
!pip install -q -r "$ROOT/requirements.txt"
sys.path.insert(0, str(ROOT))

In [ ]:
# 1) Chaves + modelos em tempo de execução — NADA é salvo no arquivo
# API KEY: cole ou Enter p/ pular. Modelo: Enter = usa padrão atual.
import os
DEFAULTS = {'GROQ_MODEL': 'openai/gpt-oss-20b',
            'OLLAMA_CLOUD_MODEL': 'nemotron-3-nano:30b-cloud',
            'OPENROUTER_MODEL': 'google/gemma-4-31b-it'}
EM_COLAB = False
try:
    from google.colab import userdata; EM_COLAB = True
except ImportError:
    pass

def pegar_segredo(nome, padrao=None):
    if EM_COLAB:
        try:
            v = userdata.get(nome)
            if v: print(f'{nome}: via Colab Secrets ✔'); return v.strip()
        except Exception: pass
    if os.getenv(nome) and not nome.endswith('_MODEL'):
        print(f'{nome}: via ambiente/.env ✔'); return os.getenv(nome)
    from getpass import getpass
    dica = f' [padrão: {padrao}]' if padrao else ''
    return getpass(f'{nome}{dica} (oculto, Enter p/ pular): ').strip()

# limpa MODELs antigos do ambiente para nunca reaproveitar cache
for m in list(DEFAULTS):
    os.environ.pop(m, None)

for k in ['GROQ_API_KEY', 'OLLAMA_CLOUD_API_KEY', 'OPENROUTER_API_KEY']:
    v = pegar_segredo(k)
    if v: os.environ[k] = v
    else: os.environ.pop(k, None)

for k, padrao in DEFAULTS.items():
    v = pegar_segredo(k, padrao)
    if v: os.environ[k] = v  # Enter vazio = usa DEFAULTS (sempre atual)

print('Modelos efetivos:', {k: os.getenv(k, v) for k, v in DEFAULTS.items()})
print('Ordem de uso: Groq → Ollama Cloud → OpenRouter.')


In [ ]:
# 2) Recarrega módulos (ignora .pyc em cache) + conecta LLM
import importlib, sys
import src.llm_client as llm_client
import src.graph as graph
importlib.reload(llm_client); importlib.reload(graph)
from src.llm_client import get_llm, effective_models
from src.graph import grade_transcricao
import pandas as pd
print('Modelos efetivos:', effective_models())
llm = get_llm()
print('LLM ativo:', type(llm).__name__)
df = pd.read_csv(ROOT / 'data/exemplos.csv')
print(df[['id','nivel']].to_string(index=False))


## Como funciona o grafo
`saudacao (>0.5?) → descoberta (>0.5?) → apresentacao (>0.5?) → fechamento → final (pesos 0.15/0.30/0.30/0.25)` → veredito `≥7 aprovado, 5-7 atenção, <5 reprovado`.

In [ ]:
# 3) Avalia os 3 exemplos (bom/médio/ruim)
for _, row in df.iterrows():
    r = grade_transcricao(row['transcricao'], llm)
    print(f"\n=== {row['id']} ({row['nivel']}) → Final {r['final_score']:.1f} [{r['veredito']}] ===")
    for k in ['saudacao','descoberta','apresentacao','fechamento']:
        print(f"  {k:13s} {r[k+'_score']*10:4.1f} — {r[k+'_feedback'][:160]}")

## Sua vez ✍️
Troque o texto abaixo pela sua transcrição e execute. Dica: após testar, limpe saídas (`Editar → Limpar todas as saídas`) antes de commitar.

In [ ]:
minha_transcricao = """Vendedor: Olá, tudo bem? Aqui é da TechSoluções... (cole aqui sua transcrição)"""
r = grade_transcricao(minha_transcricao, llm)
print('Final:', r['final_score'], r['veredito'])
for k in ['saudacao','descoberta','apresentacao','fechamento']:
    print(f"{k}: {r[k+'_score']*10:.1f} — {r[k+'_feedback']}")